## Cargar y configurar el modelo Falcon-7B-Instruct para generación de texto

In [39]:
# Importar bibliotecas necesarias
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

# Nombre del modelo
modelo_nombre = "tiiuae/falcon-7b-instruct"

# Cargar el tokenizador
tokenizer = AutoTokenizer.from_pretrained(modelo_nombre)

# Cargar el modelo
modelo = AutoModelForCausalLM.from_pretrained(
    modelo_nombre,
    device_map="auto",
    torch_dtype=torch.float16
)

# Crear el generador de texto
generator = pipeline(
    "text-generation",
    model=modelo,
    tokenizer=tokenizer,
    max_new_tokens=400,
    temperature=0.7
)

c:\Users\nunoc\OneDrive\Área de Trabalho\MIA\Generative AI\u1\gia-venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nunoc\.cache\huggingface\hub\models--tiiuae--falcon-7b-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.51s/it]

## Solicitar preferencias de viaje al usuario

In [40]:
# Pedir al usuario sus preferencias de viaje
destino = input("Introduce tu destino de viaje: ")
dias = input("Introduce la duración del viaje (en días): ")
presupuesto = input("Introduce tu presupuesto aproximado: ")
intereses = input("Introduce tus intereses (por ejemplo: cultura, gastronomía, naturaleza): ")

## Generar itinerario inicial utilizando few-shot prompting y preferencias del usuario**


In [41]:
# Crear un ejemplo para few-shot prompting
ejemplo = """
Example:
Day 1: Visit the Sagrada Familia and explore the Gothic Quarter.
Day 2: Enjoy a day at Barceloneta Beach and try local tapas in the evening.
"""

prompt = f"""
You are a travel planner.
Example:
Day 1: Explore the main square, visit the cathedral, and enjoy a traditional lunch.
Day 2: Take a guided tour of local markets, visit a museum, and relax in the park.

Now, create a {dias}-day travel itinerary for {destino} with a budget of {presupuesto} euros.
Include activities related to: {intereses}.
Organize the response day by day with short and specific activities.
Do not repeat the instruction, just provide the itinerary.
"""

# Generar el itinerario
itinerario = generator(prompt)[0]["generated_text"]

# Mostrar resultado
print("\n=== Itinerario de viaje sugerido ===\n")
print(itinerario)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.



=== Itinerario de viaje sugerido ===


You are a travel planner.
Example:
Day 1: Explore the main square, visit the cathedral, and enjoy a traditional lunch.
Day 2: Take a guided tour of local markets, visit a museum, and relax in the park.

Now, create a 3-day travel itinerary for Lisbon with a budget of 1000 euros.
Include activities related to: food, culture, beach.
Organize the response day by day with short and specific activities.
Do not repeat the instruction, just provide the itinerary.
Day 1:
1. Explore the main square with a travel guide
2. Visit the cathedral
3. Visit a traditional confectionery
4. Have lunch in a local restaurant

Day 2:
1. Guided tour of local markets with a travel guide
2. Visit a museum
3. Relax in the parks
4. Visit a traditional restaurant

Day 3:
1. Visit the Roman ruins of Jerónimos Monastery
2. Visit the Lisbon Oceanarium
3. Take a boat tour of the Tagus river
4. Visit the Castle of São Jorge

Note: Use the keyword “free” for activities in a park, 

## Configurar Wikipedia para obtención de datos reales (RAG)

In [ ]:
import wikipedia

wikipedia.set_lang('en')

def obtener_atracciones(destino):
    try:
        pagina = wikipedia.page(destino)
        resumen = wikipedia.summary(destino, sentences=5)
        return resumen
    except:
        return "No se encontraron datos reales para este destino."
    
# Obtener datos reales para el destino
info_real = obtener_atracciones(destino)

# Prompt en inglés con contexto real y ejemplo few-shot
prompt = f"""
You are a travel planner.

Real context about {destino}:
{info_real}

Example of format:
Day 1: Explore the main square, visit the cathedral, and enjoy a traditional lunch.
Day 2: Visit local markets, explore a museum, and relax in a park.

Now, create a travel itinerary for {destino} with exactly {dias} days and a budget of {presupuesto} euros.
Include activities related to: {intereses}.
Only output {dias} days, no more.
Organize the response day by day with short and specific activities.
Do not repeat the instruction, just provide the itinerary.
"""

# Generar el itinerario con Falcon
itinerario = generator(prompt)[0]["generated_text"]

# Mostrar resultado
print("\n=== Itinerario de viaje sugerido ===\n")
print(itinerario)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.



=== Itinerario de viaje sugerido ===


You are a travel planner.

Real context about Lisbon:
Lisbon (  LIZ-bən; Portuguese: Lisboa [liʒˈβoɐ] ) is the capital and largest city of Portugal, with an estimated population of 575,739, as of 2024, within its administrative limits and 3,028,000 within the metropolis, as of 2025. Lisbon is mainland Europe's westernmost capital city (second overall after Reykjavík), and the only one along the Atlantic coast, the others (Reykjavík and Dublin) being on islands. The city lies in the western portion of the Iberian Peninsula, on the northern shore of the River Tagus. The western portion of its metro area, the Portuguese Riviera, hosts the westernmost point of Continental Europe, culminating at Cabo da Roca.
Lisbon is one of the oldest cities in the world and the second-oldest European capital city (after Athens), predating other modern European capitals by centuries.

Example of format:
Day 1: Explore the main square, visit the cathedral, and enjoy 

## Filtrar y limpiar el itinerario generado, limitando el número de días

In [ ]:
def limpiar_itinerario(texto): 
    inicio = texto.rfind("Day 1")
    if inicio == -1:
        inicio = texto.rfind("Día 1")
    
    if inicio != -1:
        texto = texto[inicio:]
    
    lineas_limpias = []
    for linea in texto.split("\n"):
        linea_strip = linea.strip()
        if not linea_strip:
            continue
        if "include activities" in linea_strip.lower():
            continue
        if "organize the response" in linea_strip.lower():
            continue
        if "do not repeat" in linea_strip.lower():
            continue
        lineas_limpias.append(linea_strip)
    
    return "\n".join(lineas_limpias)

def cortar_dias_extra(texto, dias):
    linhas = texto.split("\n")
    resultado = []
    contador_dias = 0
    for linha in linhas:
        if linha.lower().startswith("day"):
            contador_dias += 1
        if contador_dias > dias:
            break
        resultado.append(linha)
    return "\n".join(resultado)

# Limpar y cortar en un solo flujo
itinerario_limpio = limpiar_itinerario(itinerario)
itinerario_final = cortar_dias_extra(itinerario_limpio, int(dias))

print("\n=== Itinerario de viaje limpio ===\n")
print(itinerario_final)



=== Itinerario de viaje limpio ===

Day 1: Visit the main square, visit the cathedral, and enjoy a traditional lunch
- Arrive in Lisbon and check-in at your accommodation.
- Visit the main square (Praça do Comércio) and admire the architecture.
- Visit the stunning St. George's Basilica, a 12th-century Romanesque basilica.
- Have lunch at a traditional bistro in the square.
- After lunch, take a stroll in the surrounding streets and squares, and visit a few shops in the commercial area.
- Relax at a nearby park or cafe.
Day 2: Visit local markets, explore a museum, and relax in a park
- Start your day by walking to the nearby food market (Mercado da Ribeira), where you can try traditional Portuguese dishes and buy fresh ingredients for your meals.
- After breakfast, explore the Lisbon Castle (Castelo de Sao Jorge), which is one of the most iconic attractions of the city.
- Have lunch in a traditional restaurant in the Mouraria neighborhood.
- Visit the Lisbon Oceanarium, one of the la

## Conclusión

En este trabajo desarrollé un planificador de viajes personalizado usando un modelo de lenguaje (LLM) y técnicas de personalización. El objetivo fue generar itinerarios realistas según mis preferencias como usuaria, cumpliendo con el enunciado del caso práctico.

Tuve varios desafíos: al inicio probé modelos que repetían el prompt o daban resultados pobres, y algunos eran “gated” y pedían permisos. Probé opciones como flan-t5 y Mistral, pero finalmente elegí tiiuae/falcon-7b-instruct porque es gratuito, abierto y me dio mejores resultados. Otro reto fue que el modelo a veces generaba más días de los que yo pedía, lo que solucioné con funciones de limpieza y corte.

Usé prompting con ejemplo (few-shot) y RAG con Wikipedia para añadir información real de cada destino. Así, el modelo no solo inventaba actividades, sino que las basaba en datos reales. También añadí control de calidad filtrando instrucciones innecesarias y limitando el número de días.

Al final, conseguí un flujo completo: introduzco mis datos, el sistema busca contexto real, genera el itinerario y lo muestra limpio y listo. Aunque no hice fine-tuning ni GAN/RL, cumplí la mayoría de los requisitos y dejé el proyecto funcionando, incluso con una interfaz en Streamlit.